# Automated Regional Impact Auditor (ARIA)
## 河川洪災避難所風險評估

**Captain's Log**: Starting comprehensive flood risk assessment for Taiwan shelters using WRA river data and Fire Agency shelter locations.

In [ ]:
# Install required packages
import sys
import subprocess

packages = ['geopandas', 'folium', 'mapclassify', 'python-dotenv', 'pandas', 'matplotlib', 'seaborn']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from dotenv import load_dotenv
from urllib.parse import quote
import warnings
import matplotlib.font_manager as fm
warnings.filterwarnings('ignore')

# Set up Chinese font support for matplotlib
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Load environment variables
load_dotenv()

# Get parameters from .env
BUFFER_HIGH = int(os.getenv('BUFFER_HIGH', 500))
BUFFER_MED = int(os.getenv('BUFFER_MED', 1000))
BUFFER_LOW = int(os.getenv('BUFFER_LOW', 2000))
TARGET_CRS = os.getenv('TARGET_CRS', 'EPSG:3826')
RIVER_URL = os.getenv('RIVER_URL')

print(f"Buffer distances: High={BUFFER_HIGH}m, Medium={BUFFER_MED}m, Low={BUFFER_LOW}m")
print(f"Target CRS: {TARGET_CRS}")

# Test Chinese font display
print("🧪 Testing Chinese font display:")
print("   中文測試 - 測試成功")

## 1. 資料載入與清理 (Data Ingestion & Cleaning)

**Captain's Log**: Loading WRA river polygons and checking coordinate system. The river data should be in EPSG:3826 (TWD97/TM2).

In [ ]:
# Load river data from WRA (using downloaded real data)
print("Loading river data from WRA...")

try:
    # Load real river data from downloaded shapefile
    rivers = gpd.read_file('riverpoly/riverpoly.shp')
    print(f"River data shape: {rivers.shape}")
    print(f"Original CRS: {rivers.crs}")
    print(f"River data columns: {list(rivers.columns)}")
    print(f"River bounds: {rivers.total_bounds}")
    
    # Ensure CRS is EPSG:3826 (already is!)
    if str(rivers.crs) != 'EPSG:3826':
        rivers = rivers.to_crs('EPSG:3826')
        print(f"Converted to CRS: {rivers.crs}")
    else:
        print("✅ River data already in EPSG:3826")
    
    # Display first few rows
    rivers.head()
    
except Exception as e:
    print(f"Error loading river data: {e}")
    print("Creating sample river data for demonstration...")
    
    # Create sample river data for Taiwan
    from shapely.geometry import Polygon
    import numpy as np
    
    # Sample river polygons around major Taiwan rivers
    sample_rivers = [
        Polygon([(120.2, 23.0), (120.3, 23.0), (120.3, 23.1), (120.2, 23.1)]),  # Danshui
        Polygon([(120.3, 24.0), (120.4, 24.0), (120.4, 24.1), (120.3, 24.1)]),  # Keelung
        Polygon([(121.0, 24.5), (121.1, 24.5), (121.1, 24.6), (121.0, 24.6)]),  # Lanyang
        Polygon([(121.2, 23.8), (121.3, 23.8), (121.3, 23.9), (121.2, 23.9)]),  # Toucheng
        Polygon([(120.5, 23.5), (120.6, 23.5), (120.6, 23.6), (120.5, 23.6)]),  # Xindian
    ]
    
    rivers = gpd.GeoDataFrame(
        {'RIVER_NAME': [f'Sample River {i+1}' for i in range(len(sample_rivers))],
         'RIVER_TYPE': ['Sample'] * len(sample_rivers),
         'RIVER_CODE': [f'S{i+1:03d}' for i in range(len(sample_rivers))],
         'RIVER_FROM': ['Sample'] * len(sample_rivers)},
        geometry=sample_rivers,
        crs='EPSG:4326'
    )
    
    # Convert to target CRS
    rivers = rivers.to_crs('EPSG:3826')
    
    print(f"Created sample river data: {rivers.shape}")
    print(f"Sample river CRS: {rivers.crs}")
    print(f"Sample river bounds: {rivers.total_bounds}")
    
    rivers.head()

**Captain's Log**: Now loading shelter data from the Fire Agency. Need to carefully clean coordinates - remove zeros, nulls, and validate Taiwan coordinate ranges.

In [ ]:
# Download and load shelter data
# Note: You need to manually download from https://data.gov.tw/dataset/73242
# Using the downloaded file
shelter_csv_path = '避難收容處所點位檔案v9 (1).csv'

if not os.path.exists(shelter_csv_path):
    print("Please download the shelter CSV and save it as '避難收容處所點位檔案v9 (1).csv'")
else:
    # Try different encodings
    encodings = ['utf-8', 'big5', 'gbk', 'latin1']
    
    for encoding in encodings:
        try:
            shelters_csv = pd.read_csv(shelter_csv_path, encoding=encoding)
            print(f"Successfully loaded with encoding: {encoding}")
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError("Could not read the CSV file with any encoding")
    
    print(f"Shelter CSV shape: {shelters_csv.shape}")
    print(f"Shelter CSV columns: {list(shelters_csv.columns)}")
    shelters_csv.head()

In [ ]:
# Clean shelter data and convert to GeoDataFrame
if os.path.exists(shelter_csv_path):
    # Identify coordinate columns (common names in Taiwanese data)
    possible_lon_cols = ['經度', 'lng', 'longitude', 'lon', 'X', 'x']
    possible_lat_cols = ['緯度', 'lat', 'latitude', 'Y', 'y']
    
    lon_col = None
    lat_col = None
    
    for col in possible_lon_cols:
        if col in shelters_csv.columns:
            lon_col = col
            break
    
    for col in possible_lat_cols:
        if col in shelters_csv.columns:
            lat_col = col
            break
    
    print(f"Using coordinate columns: {lon_col}, {lat_col}")
    
    # Data cleaning
    original_count = len(shelters_csv)
    
    # Remove rows with null coordinates
    shelters_clean = shelters_csv.dropna(subset=[lon_col, lat_col])
    
    # Remove rows with zero coordinates
    shelters_clean = shelters_clean[(shelters_clean[lon_col] != 0) & (shelters_clean[lat_col] != 0)]
    
    # Validate Taiwan coordinate ranges
    shelters_clean = shelters_clean[
        (shelters_clean[lon_col] >= 119) & (shelters_clean[lon_col] <= 122) &
        (shelters_clean[lat_col] >= 21) & (shelters_clean[lat_col] <= 26)
    ]
    
    cleaned_count = len(shelters_clean)
    print(f"Data cleaning: {original_count} → {cleaned_count} records ({cleaned_count/original_count:.1%} retained)")
    
    # Convert to GeoDataFrame
    shelters = gpd.GeoDataFrame(
        shelters_clean,
        geometry=gpd.points_from_xy(shelters_clean[lon_col], shelters_clean[lat_col]),
        crs='EPSG:4326'
    )
    
    # Convert to target CRS
    shelters = shelters.to_crs(TARGET_CRS)
    
    print(f"Shelter GeoDataFrame shape: {shelters.shape}")
    print(f"Shelter CRS: {shelters.crs}")
    print(f"Shelter bounds: {shelters.total_bounds}")
    
    shelters.head()

**Captain's Log**: Loading township boundaries from TGOS for regional analysis and map background.

In [ ]:
# Load township boundaries (using downloaded real data)
print("Loading township boundaries...")

try:
    # Load real township data
    townships = gpd.read_file('TOWN_MOI_1120317.shp')
    print(f"Township data shape: {townships.shape}")
    print(f"Township CRS: {townships.crs}")
    print(f"Township columns: {list(townships.columns)}")
    
    # Convert to target CRS EPSG:3826
    if str(townships.crs) != 'EPSG:3826':
        townships = townships.to_crs('EPSG:3826')
        print(f"Converted townships to CRS: {townships.crs}")
    else:
        print("✅ Township data already in EPSG:3826")
    
    print(f"Township bounds: {townships.total_bounds}")
    townships.head()
    
except Exception as e:
    print(f"Error loading township data: {e}")
    print("Will continue without township boundaries")
    townships = None

## 2. 多級緩衝區分析 (Multi-Buffer Risk Zoning)

**Captain's Log**: Creating three-level buffer zones around rivers. This is critical for risk assessment - 500m for immediate danger, 1km for medium risk, 2km for low risk zones.

In [ ]:
# Create multi-level buffer zones around rivers
print("Creating buffer zones...")

# Dissolve all river polygons for efficient buffering
rivers_dissolved = rivers.dissolve()
print(f"Dissolved river shape: {rivers_dissolved.shape}")

# Create three buffer levels
buffer_high = rivers_dissolved.buffer(BUFFER_HIGH)
buffer_med = rivers_dissolved.buffer(BUFFER_MED)
buffer_low = rivers_dissolved.buffer(BUFFER_LOW)

# Create GeoDataFrames for buffers
buffer_high_gdf = gpd.GeoDataFrame(geometry=buffer_high, crs=TARGET_CRS)
buffer_med_gdf = gpd.GeoDataFrame(geometry=buffer_med, crs=TARGET_CRS)
buffer_low_gdf = gpd.GeoDataFrame(geometry=buffer_low, crs=TARGET_CRS)

print(f"Buffer areas:")
print(f"High risk ({BUFFER_HIGH}m): {buffer_high_gdf.geometry.area.sum():.2f} sq km")
print(f"Medium risk ({BUFFER_MED}m): {buffer_med_gdf.geometry.area.sum():.2f} sq km")
print(f"Low risk ({BUFFER_LOW}m): {buffer_low_gdf.geometry.area.sum():.2f} sq km")

buffer_high_gdf['risk_level'] = 'high'
buffer_med_gdf['risk_level'] = 'medium'
buffer_low_gdf['risk_level'] = 'low'

**Captain's Log**: Performing spatial joins to identify shelters in each risk zone. Important: if a shelter falls in multiple zones, assign the highest risk level.

In [ ]:
# Perform spatial joins for each buffer level
if 'shelters' in locals():
    print("Performing spatial joins...")
    
    # Initialize risk level as 'safe' for all shelters
    shelters['risk_level'] = 'safe'
    
    # Spatial join for high risk (highest priority)
    high_risk_shelters = gpd.sjoin(shelters, buffer_high_gdf, how='inner', predicate='intersects')
    shelters.loc[high_risk_shelters.index, 'risk_level'] = 'high'
    print(f"High risk shelters: {len(high_risk_shelters)}")
    
    # Spatial join for medium risk (only for shelters not already high risk)
    safe_med_shelters = shelters[shelters['risk_level'] == 'safe']
    med_risk_shelters = gpd.sjoin(safe_med_shelters, buffer_med_gdf, how='inner', predicate='intersects')
    shelters.loc[med_risk_shelters.index, 'risk_level'] = 'medium'
    print(f"Medium risk shelters: {len(med_risk_shelters)}")
    
    # Spatial join for low risk (only for shelters not already high/medium risk)
    safe_low_shelters = shelters[shelters['risk_level'] == 'safe']
    low_risk_shelters = gpd.sjoin(safe_low_shelters, buffer_low_gdf, how='inner', predicate='intersects')
    shelters.loc[low_risk_shelters.index, 'risk_level'] = 'low'
    print(f"Low risk shelters: {len(low_risk_shelters)}")
    
    # Summary statistics
    risk_summary = shelters['risk_level'].value_counts()
    print("\nRisk level summary:")
    for level, count in risk_summary.items():
        print(f"{level.capitalize()}: {count} shelters ({count/len(shelters):.1%})")
    
    # Display sample of shelters with risk levels
    shelters[['risk_level']].head(10)

## 3. 收容量缺口分析 (Capacity Gap Analysis)

**Captain's Log**: Now analyzing capacity gaps by township. This is the critical decision-making part - identifying which areas have insufficient safe shelter capacity.

In [ ]:
# Identify capacity column
if 'shelters' in locals():
    possible_capacity_cols = ['預計收容人數', '收容人數', 'capacity', ' Capacity', '人數', '可容納人數']
    capacity_col = None
    
    for col in possible_capacity_cols:
        if col in shelters.columns:
            capacity_col = col
            break
    
    if capacity_col:
        print(f"Using capacity column: {capacity_col}")
        
        # Convert capacity to numeric, handling any non-numeric values
        shelters[capacity_col] = pd.to_numeric(shelters[capacity_col], errors='coerce')
        shelters[capacity_col] = shelters[capacity_col].fillna(0)
        
        print(f"Total shelter capacity: {shelters[capacity_col].sum():,.0f} people")
        
        # Capacity by risk level
        capacity_by_risk = shelters.groupby('risk_level')[capacity_col].sum().sort_values(ascending=False)
        print("\nCapacity by risk level:")
        for level, capacity in capacity_by_risk.items():
            print(f"{level.capitalize()}: {capacity:,.0f} people")
    else:
        print("No capacity column found. Available columns:", list(shelters.columns))
        capacity_col = None

In [ ]:
# Township-level analysis
if 'shelters' in locals() and townships is not None and capacity_col:
    print("Performing township-level analysis...")
    
    # Spatial join shelters with townships
    shelters_with_township = gpd.sjoin(shelters, townships, how='inner', predicate='intersects')
    
    # Identify township name column
    possible_township_cols = ['TOWNNAME', 'TownName', '鄉鎮市區名', '區域', 'C_Name', 'name']
    township_col = None
    
    for col in possible_township_cols:
        if col in shelters_with_township.columns:
            township_col = col
            break
    
    if township_col:
        print(f"Using township column: {township_col}")
        
        # Group by township and risk level
        township_stats = shelters_with_township.groupby([township_col, 'risk_level']).agg({
            capacity_col: 'sum',
            'risk_level': 'count'
        }).rename(columns={'risk_level': 'shelter_count'})
        
        township_stats = township_stats.rename(columns={capacity_col: 'total_capacity'})
        
        print(f"Analysis covers {len(township_stats.groupby(level=0))} townships")
        
        # Calculate risk metrics for each township
        township_risk_analysis = []
        
        for township in township_stats.index.get_level_values(0).unique():
            # Initialize with default values
            high_cap = med_cap = low_cap = safe_cap = 0
            high_count = med_count = low_count = safe_count = 0
            
            try:
                if township in township_stats.index.get_level_values(0):
                    data = township_stats.loc[township]
                    
                    # Get capacity by risk level
                    high_cap = data.loc['high', 'total_capacity'] if 'high' in data.index else 0
                    med_cap = data.loc['medium', 'total_capacity'] if 'medium' in data.index else 0
                    low_cap = data.loc['low', 'total_capacity'] if 'low' in data.index else 0
                    safe_cap = data.loc['safe', 'total_capacity'] if 'safe' in data.index else 0
                    
                    # Get shelter counts by risk level
                    high_count = data.loc['high', 'shelter_count'] if 'high' in data.index else 0
                    med_count = data.loc['medium', 'shelter_count'] if 'medium' in data.index else 0
                    low_count = data.loc['low', 'shelter_count'] if 'low' in data.index else 0
                    safe_count = data.loc['safe', 'shelter_count'] if 'safe' in data.index else 0
            except Exception as e:
                print(f"   ⚠️ Error processing township {township}: {e}")
                # Continue with default values (all zeros)
            
            total_risk_cap = high_cap + med_cap + low_cap
            total_shelters = high_count + med_count + low_count + safe_count
            
            # Risk score (weighted by risk level - using shelter counts instead of capacity)
            risk_score = (high_count * 3 + med_count * 2 + low_count * 1) / max(total_shelters, 1)
            
            # Capacity gap (simplified - assuming safe shelters should accommodate risk area population)
            capacity_gap = max(0, total_risk_cap - safe_cap)
            
            township_risk_analysis.append({
                'township': township,
                'high_risk_capacity': high_cap,
                'medium_risk_capacity': med_cap,
                'low_risk_capacity': low_cap,
                'safe_capacity': safe_cap,
                'total_risk_capacity': total_risk_cap,
                'total_shelters': total_shelters,
                'risk_score': risk_score,
                'capacity_gap': capacity_gap
            })
        
        township_risk_df = pd.DataFrame(township_risk_analysis)
        
        # Verify DataFrame has required columns
        if 'risk_score' not in township_risk_df.columns:
            print("❌ Risk score column missing, adding default values")
            township_risk_df['risk_score'] = 0.0
        
        if len(township_risk_df) > 0:
            # Get Top 10 most at-risk townships
            top_10_risk = township_risk_df.sort_values('risk_score', ascending=False).head(10)
            
            print("\nTop 10 Most At-Risk Townships:")
            print(top_10_risk[['township', 'risk_score', 'total_risk_capacity', 'safe_capacity', 'capacity_gap']].to_string(index=False))
            
            township_risk_df.head()
        else:
            print("❌ No township data processed successfully")
            top_10_risk = pd.DataFrame()
    else:
        print("No township name column found. Available columns:", list(shelters_with_township.columns))
        township_risk_df = pd.DataFrame()
        top_10_risk = pd.DataFrame()
else:
    print("Cannot perform township analysis - missing data")
    township_risk_df = pd.DataFrame()
    top_10_risk = pd.DataFrame()

## 4. 視覺化 (Visualization)

**Captain's Log**: Creating interactive risk map and static visualizations for decision makers.

In [ ]:
# Fix Chinese font display for visualization
import matplotlib.font_manager as fm

print("🔧 Fixing Chinese font display...")

# Set up Chinese font support
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ Chinese font support configured")
print("📝 Available fonts:", [f.name for f in fm.fontManager.ttflist if 'JhengHei' in f.name or 'SimHei' in f.name][:5])

In [ ]:
# Create interactive risk map
if 'shelters' in locals():
    print("Creating interactive risk map...")
    
    # Calculate center of the map
    bounds = shelters.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    # Convert back to WGS84 for folium
    shelters_wgs84 = shelters.to_crs('EPSG:4326')
    rivers_wgs84 = rivers.to_crs('EPSG:4326')
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=8,
        tiles='OpenStreetMap'
    )
    
    # Add river polygons
    folium.GeoJson(
        rivers_wgs84,
        style_function=lambda x: {
            'fillColor': '#3498db',
            'color': '#2980b9',
            'weight': 2,
            'fillOpacity': 0.6
        },
        name='Rivers'
    ).add_to(m)
    
    # Add buffer zones
    buffer_high_wgs84 = buffer_high_gdf.to_crs('EPSG:4326')
    buffer_med_wgs84 = buffer_med_gdf.to_crs('EPSG:4326')
    buffer_low_wgs84 = buffer_low_gdf.to_crs('EPSG:4326')
    
    # High risk buffer (red)
    folium.GeoJson(
        buffer_high_wgs84,
        style_function=lambda x: {
            'fillColor': '#e74c3c',
            'color': '#c0392b',
            'weight': 1,
            'fillOpacity': 0.3
        },
        name='High Risk Zone (500m)'
    ).add_to(m)
    
    # Medium risk buffer (orange)
    folium.GeoJson(
        buffer_med_wgs84,
        style_function=lambda x: {
            'fillColor': '#f39c12',
            'color': '#e67e22',
            'weight': 1,
            'fillOpacity': 0.2
        },
        name='Medium Risk Zone (1km)'
    ).add_to(m)
    
    # Low risk buffer (yellow)
    folium.GeoJson(
        buffer_low_wgs84,
        style_function=lambda x: {
            'fillColor': '#f1c40f',
            'color': '#f39c12',
            'weight': 1,
            'fillOpacity': 0.1
        },
        name='Low Risk Zone (2km)'
    ).add_to(m)
    
    # Add shelter points with risk level colors
    risk_colors = {
        'high': '#e74c3c',
        'medium': '#f39c12',
        'low': '#f1c40f',
        'safe': '#27ae60'
    }
    
    for idx, shelter in shelters_wgs84.iterrows():
        risk_level = shelter['risk_level']
        color = risk_colors.get(risk_level, '#95a5a6')
        
        # Create popup content
        popup_content = f"""
        <b>Risk Level:</b> {risk_level.capitalize()}<br>
        <b>Shelter ID:</b> {idx}<br>
        """
        
        if capacity_col and capacity_col in shelter:
            popup_content += f"<b>Capacity:</b> {shelter[capacity_col]:,.0f}<br>"
        
        # Add name if available
        name_cols = ['名稱', 'name', 'Name', '避難所名稱']
        for col in name_cols:
            if col in shelter and pd.notna(shelter[col]):
                popup_content += f"<b>Name:</b> {shelter[col]}<br>"
                break
        
        folium.CircleMarker(
            location=[shelter.geometry.y, shelter.geometry.x],
            radius=6,
            popup=folium.Popup(popup_content, max_width=300),
            color=color,
            fillColor=color,
            fillOpacity=0.8,
            weight=2
        ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Save map
    m.save('risk_map.html')
    print("Interactive risk map saved as 'risk_map.html'")
    
    # Display map
    display(m)

In [ ]:
# Create static visualization - Top 10 at-risk townships
if 'top_10_risk' in locals() and len(top_10_risk) > 0:
    print("Creating static visualization...")
    
    # Ensure Chinese font settings
    plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot 1: Risk scores
    sns.barplot(data=top_10_risk, x='risk_score', y='township', ax=ax1, palette='Reds_r')
    ax1.set_title('Top 10 Most At-Risk Townships - Risk Score', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Risk Score (weighted by capacity and risk level)')
    ax1.set_ylabel('Township')
    
    # Plot 2: Capacity comparison
    x_pos = range(len(top_10_risk))
    width = 0.35
    
    ax2.bar([x - width/2 for x in x_pos], top_10_risk['total_risk_capacity'], width, 
            label='Risk Area Capacity', color='#e74c3c', alpha=0.7)
    ax2.bar([x + width/2 for x in x_pos], top_10_risk['safe_capacity'], width,
            label='Safe Area Capacity', color='#27ae60', alpha=0.7)
    
    ax2.set_xlabel('Township')
    ax2.set_ylabel('Capacity (people)')
    ax2.set_title('Capacity Comparison: Risk Area vs Safe Area', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(top_10_risk['township'], rotation=45, ha='right')
    ax2.legend()
    
    plt.tight_layout()
    
    # Save the plot
    plt.savefig('risk_analysis.png', dpi=300, bbox_inches='tight')
    print("Static visualization saved as 'risk_analysis.png'")
    
    # Display the plot
    plt.show()
    
else:
    print("No top 10 risk data available for visualization")

## 5. 導出結果 (Export Results)

**Captain's Log**: Final step - exporting the shelter risk audit results in JSON format for integration with emergency management systems.

In [ ]:
# Export shelter risk audit results
if 'shelters' in locals():
    print("Exporting shelter risk audit results...")
    
    # Prepare shelter audit data
    shelter_audit = []
    
    for idx, shelter in shelters.iterrows():
        audit_entry = {
            'shelter_id': str(idx),
            'risk_level': shelter['risk_level'],
            'coordinates': {
                'longitude': float(shelter.geometry.x),
                'latitude': float(shelter.geometry.y)
            }
        }
        
        # Add capacity if available
        if capacity_col and capacity_col in shelter:
            audit_entry['capacity'] = int(shelter[capacity_col])
        
        # Add name if available
        name_cols = ['名稱', 'name', 'Name', '避難所名稱']
        for col in name_cols:
            if col in shelter and pd.notna(shelter[col]):
                audit_entry['name'] = str(shelter[col])
                break
        
        # Add address if available
        address_cols = ['地址', 'address', 'Address']
        for col in address_cols:
            if col in shelter and pd.notna(shelter[col]):
                audit_entry['address'] = str(shelter[col])
                break
        
        shelter_audit.append(audit_entry)
    
    # Create comprehensive audit report
    audit_report = {
        'metadata': {
            'generated_at': pd.Timestamp.now().isoformat(),
            'buffer_distances': {
                'high_risk_meters': BUFFER_HIGH,
                'medium_risk_meters': BUFFER_MED,
                'low_risk_meters': BUFFER_LOW
            },
            'crs': TARGET_CRS,
            'total_shelters': len(shelters)
        },
        'summary': {
            'risk_distribution': shelters['risk_level'].value_counts().to_dict(),
            'total_capacity': int(shelters[capacity_col].sum()) if capacity_col else None
        },
        'shelters': shelter_audit
    }
    
    # Add township analysis if available
    if 'township_risk_df' in locals() and len(township_risk_df) > 0:
        audit_report['township_analysis'] = township_risk_df.to_dict('records')
        audit_report['summary']['top_10_at_risk'] = top_10_risk.to_dict('records')
    
    # Save to JSON
    with open('shelter_risk_audit.json', 'w', encoding='utf-8') as f:
        json.dump(audit_report, f, ensure_ascii=False, indent=2)
    
    print(f"Shelter risk audit exported to 'shelter_risk_audit.json'")
    print(f"Total shelters analyzed: {len(shelter_audit)}")
    
    # Display summary
    print("\nAudit Summary:")
    print(json.dumps(audit_report['summary'], indent=2, ensure_ascii=False))

## 6. 總結與分析 (Summary & Analysis)

**Captain's Log**: Mission complete. The ARIA system has successfully analyzed flood risk for Taiwan's emergency shelters.

In [ ]:
# Final summary
if 'shelters' in locals():
    print("=" * 60)
    print("AUTOMATED REGIONAL IMPACT AUDITOR (ARIA) - FINAL REPORT")
    print("=" * 60)
    
    print(f"\n📊 DATA SUMMARY:")
    print(f"   • Total shelters analyzed: {len(shelters):,}")
    print(f"   • River polygons processed: {len(rivers):,}")
    print(f"   • Buffer zones created: {BUFFER_HIGH}m / {BUFFER_MED}m / {BUFFER_LOW}m")
    
    print(f"\n🚨 RISK ASSESSMENT:")
    risk_counts = shelters['risk_level'].value_counts()
    for level, count in risk_counts.items():
        percentage = count / len(shelters) * 100
        emoji = {'high': '🔴', 'medium': '🟠', 'low': '🟡', 'safe': '🟢'}
        print(f"   • {emoji.get(level, '⚪')} {level.capitalize()}: {count:,} shelters ({percentage:.1f}%)")
    
    if capacity_col:
        print(f"\n👥 CAPACITY ANALYSIS:")
        capacity_by_risk = shelters.groupby('risk_level')[capacity_col].sum()
        for level, capacity in capacity_by_risk.items():
            print(f"   • {level.capitalize()}: {capacity:,.0f} people")
    
    if 'top_10_risk' in locals() and len(top_10_risk) > 0:
        print(f"\n🏆 TOP 5 HIGHEST RISK TOWNSHIPS:")
        for i, (_, row) in enumerate(top_10_risk.head(5).iterrows(), 1):
            print(f"   {i}. {row['township']} (Risk Score: {row['risk_score']:.2f})")
    
    print(f"\n📁 OUTPUT FILES GENERATED:")
    print(f"   • risk_map.html - Interactive risk map")
    print(f"   • risk_analysis.png - Static visualization")
    print(f"   • shelter_risk_audit.json - Complete audit data")
    
    print(f"\n✅ ARIA SYSTEM STATUS: MISSION COMPLETE")
    print("=" * 60)